# Statistics EDA Demo — containeer-optuna v1.0.0

Demonstrates the statistics toolkit on bundled datasets.

In [1]:
import containeer_optuna as co
import numpy as np
import pandas as pd
print(f'v{co.__version__}')

v1.0.0


In [2]:
X_iris = co.get_dataset("iris").load()
print(f"Iris: {X_iris.shape}")
X_dia, y_dia = co.get_dataset("diabetes").load()
print(f"Diabetes: {X_dia.shape}")

Iris: (150, 5)
Diabetes: (442, 10)


## 1. Descriptive statistics

In [3]:
from containeer_optuna import describe
for col in X_dia.columns[:5]:
    d = describe(X_dia[col])
    print(f"{col:10s}: mean={d[chr(109)+chr(101)+chr(97)+chr(110)]:.2f}, "
          f"std={d[chr(115)+chr(116)+chr(100)]:.2f}, "
          f"skew={d[chr(115)+chr(107)+chr(101)+chr(119)]:.3f}")

age       : mean=-0.00, std=0.05, skew=-0.231
sex       : mean=0.00, std=0.05, skew=0.127
bmi       : mean=-0.00, std=0.05, skew=0.596
bp        : mean=-0.00, std=0.05, skew=0.290
s1        : mean=-0.00, std=0.05, skew=0.377


## 2. Normality tests (Shapiro-Wilk)

In [4]:
from containeer_optuna import shapiro_test
for col in X_dia.columns:
    r = shapiro_test(X_dia[col])
    verdict = "normal" if r.pvalue > 0.05 else "NOT normal"
    print(f"{col:10s}: W={r.statistic:.4f}, p={r.pvalue:.4f} -> {verdict}")

age       : W=0.9824, p=0.0000 -> NOT normal
sex       : W=0.6351, p=0.0000 -> NOT normal
bmi       : W=0.9728, p=0.0000 -> NOT normal
bp        : W=0.9837, p=0.0001 -> NOT normal
s1        : W=0.9903, p=0.0051 -> NOT normal
s2        : W=0.9883, p=0.0013 -> NOT normal
s3        : W=0.9632, p=0.0000 -> NOT normal
s4        : W=0.9236, p=0.0000 -> NOT normal
s5        : W=0.9911, p=0.0096 -> NOT normal
s6        : W=0.9931, p=0.0410 -> NOT normal


## 3. Correlation matrix

In [5]:
from containeer_optuna import correlation_matrix
corr, pval = correlation_matrix(X_dia, method="pearson")
print("Correlation matrix (|r| >= 0.4):")
cols = corr.columns
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = corr.iloc[i, j]
        if abs(r) >= 0.4:
            print(f"  {cols[i]:10s} ~ {cols[j]:10s}: r={r:.3f} (p={pval.iloc[i,j]:.4f})")

Correlation matrix (|r| >= 0.4):
  bmi        ~ s4        : r=0.414 (p=0.0000)
  bmi        ~ s5        : r=0.446 (p=0.0000)
  s1         ~ s2        : r=0.897 (p=0.0000)
  s1         ~ s4        : r=0.542 (p=0.0000)
  s1         ~ s5        : r=0.516 (p=0.0000)
  s2         ~ s4        : r=0.660 (p=0.0000)
  s3         ~ s4        : r=-0.738 (p=0.0000)
  s4         ~ s5        : r=0.618 (p=0.0000)
  s4         ~ s6        : r=0.417 (p=0.0000)
  s5         ~ s6        : r=0.465 (p=0.0000)


## 4. One-way ANOVA (iris species vs sepal_length)

In [6]:
from containeer_optuna import one_way_anova
# Load iris with target for grouping
X_iris_cls, y_iris_cls = co.get_dataset("iris_classification").load()
groups = [X_iris_cls["sepal_length"][y_iris_cls == g].values for g in sorted(y_iris_cls.unique())]
r = one_way_anova(*groups)
print(f"F={r.statistic:.2f}, p={r.pvalue:.6f}")
print(f"Significant: {r.pvalue < 0.05}")

F=119.26, p=0.000000
Significant: True


## Summary

The statistics module provides a uniform StatResult interface across all tests. Also available via CLI: `containeer-optuna stats describe iris`